In [0]:
import dataiku

client = dataiku.api_client()

# Retrieve global variables (accessible to all users)
global_vars = client.get_global_variables()

global_vars

In [0]:
# Check for common naming conventions like 'dss_url', 'public_url', or 'instance_url'
global_vars.get("dss_url")


In [0]:
global_vars.get("public_url")

In [0]:
import socket
import logging
from urllib.parse import urlparse
import dataiku

# Set up logging for clarity in the Job execution logs
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def get_dss_node_fqdn(client=None):
    """
    Retrieves the FQDN (Fully Qualified Domain Name) of the current Dataiku DSS node.
    
    The function prioritizes the 'dssUrl' configured in the Dataiku General Settings 
    (what Dataiku thinks its address is). If that is not set, it falls back to 
    the Operating System's socket hostname.

    Args:
        client (dataikuapi.dss.client.DSSClient, optional): An authenticated DSS client.
                                                            If None, creates a default one.

    Returns:
        str: The FQDN of the host.
    """
    # 1. Establish Client Context
    if client is None:
        client = dataiku.api_client()

    fqdn = None

    # 2. Method A: "According to Dataiku" (Configured Public URL)
    # This requires Admin privileges, which is satisfied by Assumption #2
    try:
        # In DSS v12, get_general_settings() is the standard way to access global config
        general_settings = client.get_general_settings()
        raw_settings = general_settings.get_raw()
        print(raw_settings)
        
        # 'dssUrl' is the key for the configured base URL (e.g., https://dataiku.corp.com:11000)
        dss_url = raw_settings.get("studioExternalUrl")
        
        if dss_url:
            # Parse the URL to extract just the hostname (FQDN)
            parsed_url = urlparse(dss_url)
            if parsed_url.hostname:
                fqdn = parsed_url.hostname
                logger.info(f"Found FQDN from Dataiku Settings: {fqdn}")
    except Exception as e:
        # If we lack permissions or the API fails, log and proceed to fallback
        logger.warning(f"Could not retrieve Dataiku General Settings: {e}")

    # 3. Method B: "According to the Operating System" (Fallback)
    if not fqdn:
        try:
            # socket.getfqdn() returns the host name with domain if available
            fqdn = socket.getfqdn()
            logger.info(f"Found FQDN from OS Socket: {fqdn}")
        except Exception as e:
            logger.error(f"Could not retrieve FQDN from OS: {e}")
            fqdn = "unknown-host"

    return fqdn

# --- Execution Example ---
if __name__ == "__main__":
    try:
        # We can pass the client explicitly or let the function handle it
        client = dataiku.api_client()
        current_fqdn = get_dss_node_fqdn(client)
        print(f"Current Node FQDN: {current_fqdn}")
    except Exception as e:
        print(f"Failed to execute: {e}")